In [0]:
# =============================================================
# 09_streaming — Structured Streaming
# Author: oakville3456
# Branch: feature/priority8-streaming
# Purpose: Process data continuously as it arrives
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *

RAW        = "abfss://raw-landing@saretailsalesdev.dfs.core.windows.net/sales/"
BRONZE     = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/sales"
CHECKPOINT = "abfss://bronze@saretailsalesdev.dfs.core.windows.net/_checkpoint/streaming_demo"

# Define schema explicitly
schema = StructType([
    StructField("order_id",    IntegerType(), True),
    StructField("store_id",    StringType(),  True),
    StructField("product",     StringType(),  True),
    StructField("quantity",    IntegerType(), True),
    StructField("price",       DoubleType(),  True),
    StructField("order_date",  StringType(),  True),
    StructField("customer_id", StringType(),  True),
])

# Verify raw landing has files to stream
print("=== Raw landing files available ===")
raw_files = spark.read \
    .schema(schema) \
    .option("header", "true") \
    .csv(RAW)
print(f"✅ Total raw rows available : {raw_files.count()}")

# Check existing Bronze
bronze_count = spark.read.format("delta").load(BRONZE).count()
print(f"✅ Current Bronze rows      : {bronze_count}")
print()
print("✅ Ready to build streaming pipeline!")

In [0]:
# Cell 2 — Build a streaming query from Bronze Delta table
# Reading from Delta as a stream is the most common production pattern

SILVER_STREAM = "abfss://silver@saretailsalesdev.dfs.core.windows.net/sales_stream"
CHECKPOINT_STREAM = "abfss://silver@saretailsalesdev.dfs.core.windows.net/_checkpoint/silver_stream"

# Read Bronze as a stream
bronze_stream = (
    spark.readStream
    .format("delta")
    .load(BRONZE)
)

print("=== Bronze stream schema ===")
bronze_stream.printSchema()
print(f"✅ isStreaming: {bronze_stream.isStreaming}")

In [0]:
# Cell 3 — Transform stream and write to Silver_stream
from pyspark.sql import functions as F

# Transform — clean and enrich
silver_stream = (
    bronze_stream
    .filter(F.col("order_id").isNotNull())
    .filter(F.col("price") > 0)
    .withColumn("order_date",  F.try_to_date("order_date"))
    .withColumn("revenue",     F.col("quantity") * F.col("price"))
    .withColumn("_processed_at", F.current_timestamp())
)

# Write stream to Delta — trigger once (like a batch)
query = (
    silver_stream.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_STREAM)
    .trigger(availableNow=True)   # process all available now, then stop
    .start(SILVER_STREAM)
)

query.awaitTermination()

# Verify output
count = spark.read.format("delta").load(SILVER_STREAM).count()
print(f"✅ Silver stream rows written: {count}")
spark.read.format("delta").load(SILVER_STREAM).show(5)

In [0]:
# Cell 4 — Streaming aggregation — real time Gold
GOLD_STREAM     = "abfss://gold@saretailsalesdev.dfs.core.windows.net/sales_stream"
CHECKPOINT_GOLD = "abfss://gold@saretailsalesdev.dfs.core.windows.net/_checkpoint/gold_stream"

# Read Silver stream
silver_read = (
    spark.readStream
    .format("delta")
    .load(SILVER_STREAM)
)

# Aggregate by store_id in real time
gold_stream = (
    silver_read
    .filter(F.col("order_date").isNotNull())
    .groupBy("store_id", "order_date")
    .agg(
        F.sum("revenue").alias("total_revenue"),
        F.count("order_id").alias("order_count"),
        F.approx_count_distinct("customer_id").alias("unique_customers")
    )
)

# Write aggregated stream
query2 = (
    gold_stream.writeStream
    .format("delta")
    .outputMode("complete")
    .option("checkpointLocation", CHECKPOINT_GOLD)
    .trigger(availableNow=True)
    .start(GOLD_STREAM)
)

query2.awaitTermination()

# Verify
count = spark.read.format("delta").load(GOLD_STREAM).count()
print(f"✅ Gold stream rows: {count}")
spark.read.format("delta").load(GOLD_STREAM).show(10)

In [0]:
# Cell 5 — Batch vs Streaming comparison

print("=== Batch Gold (existing) ===")
batch_gold = spark.table("adb_retail_dev.gold.sales_daily")
print(f"Rows: {batch_gold.count()}")
batch_gold.orderBy("store_id", "order_date").show(5)

print("=== Streaming Gold (just built) ===")
stream_gold = spark.read.format("delta").load(GOLD_STREAM)
print(f"Rows: {stream_gold.count()}")
stream_gold.orderBy("store_id", "order_date").show(5)

print("Key differences:")
print("Batch   : runs once daily, processes all data")
print("Stream  : runs continuously, processes new data as it arrives")
print("Result  : same aggregations, different latency")
print("Batch   : data fresh as of 06:00 AM")
print("Stream  : data fresh within seconds ✅")